# Module 24 — Long-term vector memory, and the leak that ends a B2B contract

**THE ONE IDEA:** long-term memory **is retrieval** — everything from `7.rag/` applies.
The agent-specific parts are two: **episodic vs semantic**, and **per-user namespacing**.

The namespacing is not a nicety. A second user is added at the end specifically to
**prove no leakage**. Forgetting to namespace is catastrophic in any B2B or regulated
setting, and it is a one-line mistake.

Runs on `BAAI/bge-small-en-v1.5`, already cached locally — **no API key, no cost.**


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import numpy as np
from datetime import datetime, timedelta
from sentence_transformers import SentenceTransformer

enc = SentenceTransformer("BAAI/bge-small-en-v1.5")
def embed(texts): return enc.encode(texts, normalize_embeddings=True)

class MemoryStore:
    """Namespaced by user_id. THE NAMESPACE IS THE WHOLE SECURITY MODEL."""
    def __init__(self): self.rows = []          # (user_id, kind, text, when, vec)

    def add(self, user_id, kind, text, when=None):
        self.rows.append((user_id, kind, text, when or datetime.now(), embed([text])[0]))

    def search(self, user_id, query, k=3, kind=None):
        # The filter happens BEFORE similarity. Never filter after ranking.
        cand = [r for r in self.rows if r[0] == user_id and (kind is None or r[1] == kind)]
        if not cand: return []
        sims = np.array([r[4] for r in cand]) @ embed([query])[0]
        return [(cand[i][1], cand[i][2], float(sims[i])) for i in np.argsort(-sims)[:k]]

store = MemoryStore()
print("encoder ready, dim =", enc.get_embedding_dimension())

## Episodic vs semantic

Both come from the same conversation. **Episodic** is what happened, with a timestamp.
**Semantic** is the de-tensed fact left behind.

In [ ]:
now = datetime.now()
# EPISODIC — events, time-keyed
store.add("u:alice", "episodic", "Alice asked about early repayment charges on a 5-year fix.", now - timedelta(days=30))
store.add("u:alice", "episodic", "Alice was declined at 95% LTV and asked about a larger deposit.", now - timedelta(days=12))
store.add("u:alice", "episodic", "Alice sent three months of payslips.", now - timedelta(days=2))
# SEMANTIC — de-tensed facts
store.add("u:alice", "semantic", "Alice prefers email contact and concise answers.")
store.add("u:alice", "semantic", "Alice is employed, not self-employed.")

for kind, text, s in store.search("u:alice", "Why was her application refused?", k=3):
    print(f"  [{kind:8}] {s:.3f}  {text[:66]}")

## Recall into a *new* conversation

This is the whole point of long-term memory. Nothing from the old session is in the
message list — the facts are retrieved into a **fresh** context.

In [ ]:
def new_session(user_id, question):
    hits = store.search(user_id, question, k=3)
    block = "\n".join(f"- ({k}) {t}" for k, t, _ in hits)
    return (f"[memory retrieved for {user_id}]\n{block}\n\n"
            f"[new conversation, no prior messages]\nUSER: {question}")

print(new_session("u:alice", "Should I contact her by phone about the deposit?"))

## The leak test — a second user

In [ ]:
store.add("u:bob", "semantic", "Bob is self-employed with two years of accounts.")
store.add("u:bob", "episodic", "Bob asked about buy-to-let affordability.", now)

print("query 'self-employed accounts' as ALICE:")
alice = store.search("u:alice", "self-employed accounts", k=3)
for k, t, s in alice: print(f"   {s:.3f}  {t[:64]}")

print("\nquery the same as BOB:")
for k, t, s in store.search("u:bob", "self-employed accounts", k=3):
    print(f"   {s:.3f}  {t[:64]}")

leaked = [t for _, t, _ in alice if "Bob" in t]
print(f"\nBob's rows visible to Alice: {leaked or 'NONE'}")
assert not leaked, "NAMESPACE LEAK"
print("assertion passed — namespace isolation holds")

print("""
LESSON - long-term memory IS retrieval. Everything in 7.rag about chunking,
reranking and stale facts applies here unchanged. Three agent-specific points:

  EPISODIC vs SEMANTIC   'Alice was declined at 95% LTV on 12 Aug' is an event
                         you query for narrative. 'Alice is employed' is a fact
                         you query for state. Store episodic by DEFAULT and
                         distil semantic facts on a slower schedule, or your
                         fact store fills with transient observations.

  NAMESPACING            the filter runs BEFORE similarity, never after. Filter
                         after ranking and a neighbouring tenant's row can still
                         displace your own from top-k. One missing user_id is a
                         B2B incident, not a bug - which is why the assert above
                         is in the notebook rather than in a test file.

  PROVENANCE             every row should carry source and timestamp. Memory
                         retrieved into context is INDISTINGUISHABLE from a tool
                         result (module 15) - so a poisoned document that gets
                         written to memory becomes a PERSISTENT injection.
                         Validate before writing, not just before acting.

Note what this tier cannot do: return 'MX-7741' exactly. Similarity search gives
you near-misses. That is module 23's job, and the two are complements.""")

---

**Next:** `25_memory_procedural_and_forgetting.ipynb`